In [1]:
# Remove existing repo to prevent errors upon session restart
!rm -rf NLP_term_project
!git clone https://github.com/thit2003/NLP_term_project.git

Cloning into 'NLP_term_project'...
remote: Enumerating objects: 28127, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 28127 (delta 7), reused 27 (delta 5), pack-reused 28094 (from 1)
Receiving objects: 100% (28127/28127), 276.94 MiB | 10.73 MiB/s, done.
Resolving deltas: 100% (2993/2993), done.


In [2]:
import os
import pandas as pd
import numpy as np
from datasets import Dataset, concatenate_datasets, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

MODEL_NAME = "roberta-base"

LABEL2ID = {"positive": 0, "neutral": 1, "negative": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [3]:
# -------------------------
# Twitter sentiment loaders
# -------------------------
def load_twitter_train(path="/content/NLP_term_project/twitter_sentiment/train.csv") -> Dataset:
    """
    Extracts text and sentiment columns.
    Handles non-UTF8 encodings robustly.
    """
    last_err = None
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin1"):
        try:
            df = pd.read_csv(path, encoding=enc)
            break
        except UnicodeDecodeError as e:
            last_err = e
            df = None

    if df is None:
        raise last_err

    df = df[["text", "sentiment"]].rename(columns={"sentiment": "label"})
    df["text"] = df["text"].astype(str)
    df["label"] = df["label"].astype(str).str.strip().str.lower()
    df = df[df["label"].isin(LABEL2ID)].dropna(subset=["text", "label"])
    df["labels"] = df["label"].map(LABEL2ID).astype(int)

    # Store the dataframe temporarily to check class distribution later
    global tw_train_df
    tw_train_df = df

    return Dataset.from_pandas(df[["text", "labels"]])


def load_twitter_test(path="/content/NLP_term_project/twitter_sentiment/test.csv") -> Dataset:
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip() or "," not in line:
                continue
            text, label = line.rsplit(",", 1)
            text = text.strip()
            label = label.strip().lower()
            if text and label in LABEL2ID:
                rows.append({"text": text, "labels": LABEL2ID[label]})
    return Dataset.from_list(rows)


# -------------------------
# SST-2 loader
# -------------------------
def load_sst2(split: str) -> Dataset:
    """
    SST-2 labels: 0 = negative, 1 = positive.
    Maps: 1 -> positive (0), 0 -> negative (2).
    """
    sst2 = load_dataset("glue", "sst2")
    ds = sst2[split]
    rows = []
    for ex in ds:
        text = ex["sentence"]
        lab = ex["label"]
        if lab == 1:
            rows.append({"text": text, "labels": LABEL2ID["positive"]})
        elif lab == 0:
            rows.append({"text": text, "labels": LABEL2ID["negative"]})
    return Dataset.from_list(rows)

In [4]:
# 1) Load Datasets
tw_train = load_twitter_train()
tw_eval = load_twitter_test()

sst2_train = load_sst2("train")
sst2_eval = load_sst2("validation")

# 2) Downsample SST-2 to balance the Twitter dataset
sst2_pos = sst2_train.filter(lambda x: x["labels"] == LABEL2ID["positive"])
sst2_neg = sst2_train.filter(lambda x: x["labels"] == LABEL2ID["negative"])

# Sample exactly what we need to balance the ~11k neutral Twitter examples
sst2_pos_sampled = sst2_pos.shuffle(seed=42).select(range(3000))
sst2_neg_sampled = sst2_neg.shuffle(seed=42).select(range(3500))

sst2_train_balanced = concatenate_datasets([sst2_pos_sampled, sst2_neg_sampled])

# 3) Combine train/eval
train_ds = concatenate_datasets([tw_train, sst2_train_balanced])
eval_ds = concatenate_datasets([tw_eval, sst2_eval])

# 4) Diagnostic Verification
labels_array = np.array(train_ds['labels'])
unique, counts = np.unique(labels_array, return_counts=True)
total = len(labels_array)

print("\n--- FINAL BALANCED TRAIN DISTRIBUTION ---")
for u, c in zip(unique, counts):
    label_name = ID2LABEL[u]
    print(f"  {label_name}: {c} ({c/total*100:.2f}%)")
print(f"TOTAL TRAIN SIZE: {total}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]


--- FINAL BALANCED TRAIN DISTRIBUTION ---
  positive: 11582 (34.08%)
  neutral: 11118 (32.72%)
  negative: 11281 (33.20%)
TOTAL TRAIN SIZE: 33981


In [5]:
# 1) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    # Padding=False allows DataCollatorWithPadding to handle dynamic padding later
    return tokenizer(examples["text"], truncation=True, padding=False, max_length=128)

print("Tokenizing train dataset...")
tokenized_train = train_ds.map(tokenize_function, batched=True)

print("Tokenizing eval dataset...")
tokenized_eval = eval_ds.map(tokenize_function, batched=True)

# 2) Build Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

Tokenizing train dataset...


Map:   0%|          | 0/33981 [00:00<?, ? examples/s]

Tokenizing eval dataset...


Map:   0%|          | 0/4406 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()

    f1s = []
    for c in [0, 1, 2]:
        tp = np.sum((preds == c) & (labels == c))
        fp = np.sum((preds == c) & (labels != c))
        fn = np.sum((preds != c) & (labels == c))
        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        f1s.append(f1)

    return {"accuracy": float(acc), "f1_macro": float(np.mean(f1s))}

training_args = TrainingArguments(
    output_dir="./results_english",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [7]:
# Model Training
print("Setting up Trainer...")
trainer.train()
metrics = trainer.evaluate()
print("Final eval:", metrics)

# Save Model
out_dir = "./final_eng_model_v1"

model.config.id2label = ID2LABEL
model.config.label2id = LABEL2ID

trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)

print("Saved model to:", out_dir)
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

Setting up Trainer...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.458352,0.515707,0.810032,0.806229
2,0.398783,0.444550,0.830685,0.830195
3,0.329924,0.507774,0.825465,0.823479


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Final eval: {'eval_loss': 0.4449451267719269, 'eval_accuracy': 0.8306854289605083, 'eval_f1_macro': 0.8301954929470571, 'eval_runtime': 10.9927, 'eval_samples_per_second': 400.81, 'eval_steps_per_second': 12.554, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: ./final_eng_model_v1
id2label: {0: 'positive', 1: 'neutral', 2: 'negative'}
label2id: {'positive': 0, 'neutral': 1, 'negative': 2}


In [8]:
import shutil
from google.colab import drive

# Mount drive if not already mounted
drive.mount('/content/drive')

# Copy the entire folder to MyDrive
out_dir = "./final_eng_model_v1"
drive_dir = "/content/drive/MyDrive/final_eng_model_v1"

# Prevent FileExistsError if running multiple times
if os.path.exists(drive_dir):
    shutil.rmtree(drive_dir)

shutil.copytree(out_dir, drive_dir)
print(f"Successfully copied model to {drive_dir}")

Mounted at /content/drive
Successfully copied model to /content/drive/MyDrive/final_eng_model_v1
